In [36]:
"""Document loading module for RAG system.

This module provides functionality to load support article data from JSON files
and convert it into formats suitable for RAG processing, including pandas
DataFrames and LangChain Document objects.
"""

from typing import List

import pandas as pd
import json
from pathlib import Path
from langchain_core.documents import Document


class DocumentLoader:
    """
    A utility class for loading and converting JSON support article data into LangChain Document objects.

    Input Requirement:
    - The input must be a JSON file with a list of support articles.
    - Each article must contain the following fields:
        - 'id': Unique identifier for each article.
        - 'title': The title of the support article.
        - 'content': The main content or body of the article.
        - 'category': Category of the article (e.g., 'integrations', 'billing', 'troubleshooting').
        - 'help_score': Numerical score indicating article helpfulness (0.0 to 1.0).
        - 'view_count': Number of times the article has been viewed.
        - 'last_updated': Timestamp indicating when the article was last updated.

    Preferred date format for 'last_updated': 'YYYY-MM-DD' or 'YYYY-MM-DD HH:MM:SS'.

    Example JSON format:
    [
        {
            "id": "KB001",
            "title": "How to integrate with Slack",
            "content": "To integrate our project management tool with Slack...",
            "category": "integrations",
            "help_score": 0.92,
            "view_count": 1523,
            "last_updated": "2024-10-05"
        }
    ]
    """

    def __init__(self, json_file: str):
        """
        Initializes the DocumentLoader with the path to a JSON file.

        Args:
            json_file (str): Absolute or relative path to the JSON file.

        Raises:
            ValueError: If input is not a non-empty string.
        """
        if not isinstance(json_file, str) or not json_file.strip():
            raise ValueError("json_file must be a non-empty string")

        self.json_file = json_file
        self.required_columns = {
            "id",
            "title",
            "content",
            "category",
            "help_score",
            "view_count",
            "last_updated",
        }

    def load_data(self) -> pd.DataFrame:
        """
        Loads and parses support article data from the specified JSON file.

        Functionality:
        - Verifies that the file exists and is accessible.
        - Parses the JSON content into a pandas DataFrame.
        - Validates that each article contains all required fields.
        - Returns a DataFrame with all article data.

        Returns:
            pd.DataFrame: A DataFrame containing all support articles with required columns.

        Raises:
            FileNotFoundError: If the file path does not exist.
            json.JSONDecodeError: If file contains invalid JSON.
            ValueError: If required fields are missing or DataFrame is empty.
            KeyError: If required columns are not present in the data.
        """

        file_path = Path(self.json_file)

        # ---------------------------------------------------------
        # 1. Validate file
        # ---------------------------------------------------------

        if not file_path.exists():
            raise FileNotFoundError(
                f"Data file not found: {file_path}"
            )

        if not file_path.is_file():
            raise ValueError(
                f"Data path is not a file: {file_path}"
            )

        # ---------------------------------------------------------
        # 2. Load JSON
        # ---------------------------------------------------------

        try:
            with file_path.open(
                mode="r",
                encoding="utf-8"
            ) as file:
                data = json.load(file)
        except PermissionError as exc:
            raise PermissionError(
                f"Permission denied when reading: {file_path}"
            ) from exc
        except json.JSONDecodeError as exc:
            raise json.JSONDecodeError(
                f"Invalid json in {file_path}: {exc.msg}",
                exc.doc,
                exc.pos
            ) from exc

        # ---------------------------------------------------------
        # 3. Validate JSON structure
        # ---------------------------------------------------------

        if not isinstance(data, list):
            raise ValueError(
                "JSON root must be a list of support articles"
            )

        if not data:
            raise ValueError(
                "JSON file contains no support articles"
            )

        if not all(isinstance(article, dict) for article in data):
            raise ValueError(
                "Each support article must be a JSON object"
            )

        # ---------------------------------------------------------
        # 4. Create DataFrame
        # ---------------------------------------------------------

        dataframe = pd.DataFrame(data)

        if dataframe.empty:
            raise ValueError(
                "DataFrame is empty after parsing JSON"
            )

        # ---------------------------------------------------------
        # 5. Validate columns
        # ---------------------------------------------------------

        missing_columns = self.required_columns.difference(
            dataframe.columns
        )

        if missing_columns:
            raise KeyError(
                "Required columns missing from dataframe: "
                + ", ".join(sorted(missing_columns))
            )

        # ---------------------------------------------------------
        # 6. Validate required fields
        # ---------------------------------------------------------
        null_columns = [
            column
            for column in self.required_columns
            if dataframe[column].isna().any()
        ]

        if null_columns:
            raise ValueError(
                "Required columns contain null values: "
                + ", ".join(sorted(null_columns))
            )

        # ---------------------------------------------------------
        # 7. Validate string fields
        # ---------------------------------------------------------

        string_columns = {
            "id",
            "title",
            "content",
            "category",
            "last_updated",
        }

        for column in string_columns:
            invalid_values = (
                ~dataframe[column].apply(
                    lambda value: isinstance(value, str)
                )
            )

            if invalid_values.any():
                raise ValueError(
                    f"Column '{column}' must contain only str"
                )

        # ---------------------------------------------------------
        # 8. Validate numeric fields
        # ---------------------------------------------------------

        numeric_columns = {
            "help_score",
            "view_count",
        }

        for column in numeric_columns:
            converted = pd.to_numeric(
                dataframe[column],
                errors="coerce",
            )

            if converted.isna().any():
                raise ValueError(
                    f"Column '{column}' must contain numeric values"
                )

            dataframe[column] = converted

        # ---------------------------------------------------------
        # 9. Validate numeric constraints
        # ---------------------------------------------------------

        if (dataframe["view_count"] < 0).any():
            raise ValueError(
                "'view_count' cannot contain negative values"
            )

        if (
            (dataframe["help_score"] < 0) |
            (dataframe["help_score"] > 1)
        ).any():
            raise ValueError(
                "'help_score' must be between 0 and 1"
            )

        # ---------------------------------------------------------
        # 10. Validate IDs
        # ---------------------------------------------------------

        if dataframe["id"].duplicated().any():
            duplicated_ids = (
                dataframe.loc[
                    dataframe["id"].duplicated(keep=False),
                    "id"
                ]
                .unique()
                .tolist()
            )

            raise ValueError(
                f"Duplicate article IDs found: {duplicated_ids}"
            )

        # ---------------------------------------------------------
        # 11. Normalize string values
        # ---------------------------------------------------------

        for column in string_columns:
            dataframe[column] = dataframe[column].str.strip()

        # ---------------------------------------------------------
        # 12. Final validation
        # ---------------------------------------------------------

        if dataframe["id"].eq("").any():
            raise ValueError(
                "'id' cannot contain empty values"
            )

        if dataframe["title"].eq("").any():
            raise ValueError(
                "'title' cannot contain empty values"
            )

        if dataframe["content"].eq("").any():
            raise ValueError(
                "'content' cannot contain empty values"
            )

        if dataframe["category"].eq("").any():
            raise ValueError(
                "'category' cannot contain empty values"
            )

        return dataframe

    def create_documents(self, data: pd.DataFrame) -> List[Document]:
        """
        Converts validated article data from a DataFrame into LangChain Document objects.

        Args:
            data (pandas.DataFrame): A DataFrame where each row represents a support article. Required columns:
                - 'id'
                - 'title'
                - 'content'
                - 'category'
                - 'help_score'
                - 'view_count'
                - 'last_updated'

        Returns:
            List[Document]: A list of LangChain-compatible Document objects with metadata.

        Raises:
            ValueError: If required columns are missing or DataFrame is empty.
            TypeError: If input is not a pandas DataFrame.
        """

        # ---------------------------------------------------------
        # 1. Validate input
        # ---------------------------------------------------------

        if not isinstance(data, pd.DataFrame):
            raise TypeError(
                "Input must be a pandas DataFrame"
            )

        # ---------------------------------------------------------
        # 2. Validate dataframe
        # ---------------------------------------------------------

        missing_columns = self.required_columns.difference(
            data.columns
        )

        if missing_columns:
            raise KeyError(
                "Required columns missing from dataframe: "
                + ", ".join(missing_columns)
            )

        if data.empty:
            raise ValueError(
                "DataFrame is empty"
            )

        # ---------------------------------------------------------
        # 3. Create documents
        # ---------------------------------------------------------

        documents = []
        for _, row in data.iterrows():
            doc = Document(
                id=row["id"],
                title=row["title"],
                page_content=row["content"],
                category=row["category"],
                metadata={
                    "help_score": row["help_score"],
                    "view_count": row["view_count"],
                    "last_updated": row["last_updated"]
                }
            )
            documents.append(doc)

        return documents

loader = DocumentLoader("./data.json")
data = loader.load_data()
docs = loader.create_documents(data)

In [ ]:
"""Vector store module for semantic search in RAG system.

This module provides functionality to create, manage, and query vector embeddings
for semantic search. It handles embedding generation, index creation/loading,
and similarity search operations using sentence transformers.
"""

from typing import Dict, List, Tuple

import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer


class VectorStore:
    """
    A class for creating and managing vector embeddings and indexes for semantic search.

    Responsibilities:
    - Generate embeddings from article titles and content using SentenceTransformer models.
    - Build and persist vector indexes with metadata.
    - Load pre-built indexes from disk.
    - Perform efficient similarity search using cosine similarity.
    - Manage document metadata alongside embeddings.

    **Expected Fields in DataFrame:**
    The `df` parameter passed to `create_index()` should include:
    - 'id': Unique article identifier.
    - 'title': The title of the support article (used for embeddings).
    - 'content': The content of the article.
    - 'category': Article category (e.g., 'integrations', 'billing').
    - 'help_score': Helpfulness score (0.0 to 1.0).
    - 'view_count': Number of views.
    - 'last_updated': Timestamp of last update.

    **Embedding Generation:**
    Uses SentenceTransformer model `all-MiniLM-L6-v2` (384 dimensions) for embeddings.

    **Similarity Search:**
    Uses cosine similarity to find the most relevant documents for a query.
    """

    def __init__(self, model_name: str = "all-MiniLM-L6-v2"):
        """
        Initializes the VectorStore with a specified embedding model.

        Args:
            model_name (str): Name of the SentenceTransformer model to use.
        """
        self.model_name = model_name
        self.model = None

    def generate_embeddings(self, texts: List[str]) -> np.ndarray:
        """
        Generates sentence embeddings for a list of texts using SentenceTransformer.

        Args:
            texts (List[str]): A list of text strings (e.g., article titles or content).

        Returns:
            np.ndarray: Array of sentence embedding vectors (shape: [n_texts, embedding_dim]).

        Raises:
            ValueError: If `texts` is not a non-empty list.
            TypeError: If `texts` is not a list.
        """
        raise NotImplementedError("Implement embedding generation using SentenceTransformer.")

    def create_index(self, df: pd.DataFrame, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Creates a vector index from the given DataFrame and saves it to disk.

        Process:
        1. Generate embeddings for article titles (or titles + content).
        2. Create index dictionary with embeddings and metadata.
        3. Save index to disk using pickle.
        4. Return the created index.

        Args:
            df (pd.DataFrame): DataFrame containing support articles with required fields:
                - 'id': Unique article ID.
                - 'title': Article title (used for embeddings).
                - 'content': Article content.
                - 'category': Article category.
                - 'help_score': Helpfulness score.
                - 'view_count': View count.
                - 'last_updated': Last updated timestamp.
            index_file_name (str): Name of the index file (e.g., 'support_index.pkl').
            index_folder_name (str): Directory where the index will be saved.

        Returns:
            dict: The created index containing embeddings and metadata.

        Raises:
            ValueError: If DataFrame is empty or missing required columns.
            OSError: If unable to create directory or save file.
        """
        raise NotImplementedError("Implement index creation and serialization logic.")

    def load_index(self, index_file_name: str, index_folder_name: str) -> Dict:
        """
        Loads a precomputed vector index from disk.

        Args:
            index_file_name (str): The file name of the saved index (e.g., 'support_index.pkl').
            index_folder_name (str): The directory where the index is stored.

        Returns:
            dict: The loaded index containing embeddings and metadata.

        Raises:
            FileNotFoundError: If the index file does not exist.
            ValueError: If the index structure is invalid or corrupted.
            KeyError: If required keys are missing in the index.
        """
        raise NotImplementedError("Implement index loading and validation.")

    def get_query_embedding(self, query: str) -> np.ndarray:
        """
        Generates an embedding for a single query string.

        Args:
            query (str): The query string to embed.

        Returns:
            np.ndarray: The embedding vector for the query (shape: [embedding_dim]).

        Raises:
            ValueError: If `query` is not a valid non-empty string.
        """
        raise NotImplementedError("Implement query embedding using SentenceTransformer.")

    def find_top_k_matches(self, query_embedding: np.ndarray, index: Dict, k: int = 5) -> List[Tuple[int, float]]:
        """
        Finds the top-k most similar articles from the index using cosine similarity.

        Args:
            query_embedding (np.ndarray): The embedding of the input query.
            index (dict): The index containing document embeddings and metadata.
            k (int): The number of top matches to return (default: 5).

        Returns:
            list: A list of tuples (document_index, similarity_score) representing the top-k matches,
                  sorted by similarity score in descending order.

        Raises:
            ValueError: If inputs are invalid (e.g., index missing embeddings).
            TypeError: If query_embedding is not a numpy array.
        """
        raise NotImplementedError("Implement similarity search using cosine similarity.")

